http://nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf

# Import Library

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# Create SparkSession

In [2]:
spark = SparkSession.builder \
    .appName("Silver to Gold") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()
print("SparkSession created successfully.")


# Kiểm tra RAM cấu hình cho Driver
driver_memory = spark.sparkContext.getConf().get("spark.driver.memory")
print(f"Driver Memory cấu hình: {driver_memory}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/26 23:27:47 WARN Utils: Your hostname, Anle-Lenovo, resolves to a loopback address: 127.0.1.1; using 192.168.1.23 instead (on interface wlo1)
26/06/26 23:27:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/26 23:27:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession created successfully.
Driver Memory cấu hình: 4g


# Read data from Silver Layer

In [3]:
silver_df = spark.read.parquet("../data/silver/data_demo.parquet")
silver_df.show(5)
print(f"List columns in dataframe: {silver_df.columns}")
print(f"Number of columns: {len(silver_df.columns)}")
print(f"Data frame schema: {silver_df.printSchema()}")

+---------------+------------------+------------------+------------------+------------------+------------------+------------+-----------+----------+------------+---------------------+------------+---------+-------------------+-------------------+-------------------+------------+------------+--------------+
|passenger_count|  pickup_longitude|   pickup_latitude|store_and_fwd_flag| dropoff_longitude|  dropoff_latitude|payment_type|fare_amount|tip_amount|tolls_amount|improvement_surcharge|total_amount|vendor_id|    pickup_datetime|   dropoff_datetime|trip_distance_miles|rate_code_id|extra_amount|mta_tax_amount|
+---------------+------------------+------------------+------------------+------------------+------------------+------------+-----------+----------+------------+---------------------+------------+---------+-------------------+-------------------+-------------------+------------+------------+--------------+
|              1|-73.97879028320312|40.745174407958984|                 N|-7

In [4]:
silver_df.count()

11382048

# dim_date

In [5]:
dim_date = (
    silver_df
    .select(
        F.to_date("pickup_datetime").alias("full_date")
    )
    .distinct()
    .withColumn(
        "date_key",
        F.date_format("full_date", "yyyyMMdd").cast("int")
    )
    .withColumn(
        "year",
        F.year("full_date")
    )
    .withColumn(
        "month",
        F.month("full_date")
    )
    .withColumn(
        "day",
        F.dayofmonth("full_date")
    )
    .withColumn(
        "weekday",
        F.dayofweek("full_date")
    )
    .withColumn(
        "quarter",
        F.quarter("full_date")
    )
    .withColumn(
        "is_weekend",
        F.dayofweek("full_date").isin([1, 7])
    )
)

# dim_vendor

In [6]:
dim_vendor = (
    silver_df
    .select("vendor_id")
    .distinct()
    .withColumn(
        "vendor_name",
        F.when(F.col("vendor_id") == 1, "Creative Mobile Technologies")
         .when(F.col("vendor_id") == 2, "VeriFone Inc")
         .otherwise("Unknown")
    )
)

# dim_payment

In [7]:
dim_payment = (
    silver_df
    .select("payment_type")
    .distinct()
    .withColumn(
        "payment_name",
        F.when(F.col("payment_type") == 1, "Credit Card")        
         .when(F.col("payment_type") == 2, "Cash")
         .when(F.col("payment_type") == 3, "No Charge")
         .when(F.col("payment_type") == 4, "Dispute")
         .when(F.col("payment_type") == 5, "Unknown")
         .when(F.col("payment_type") == 6, "Voided Trip")
         .otherwise("Other")
    )
)

# dim_rate_code

In [8]:
dim_rate_code = (
    silver_df
    .select("rate_code_id")
    .distinct()
    .withColumn(
        "rate_code_name",
        F.when(F.col("rate_code_id") == 1, "Standard Rate")
         .when(F.col("rate_code_id") == 2, "JFK")
         .when(F.col("rate_code_id") == 3, "Newark")
         .when(F.col("rate_code_id") == 4, "Nassau or Westchester")
         .when(F.col("rate_code_id") == 5, "Negotiated Fare")
         .when(F.col("rate_code_id") == 6, "Group Ride")
         .otherwise("Other")
    )
)

# fact_trips

In [9]:
fact_trips = (
    silver_df

    # Surrogate key
    .withColumn(
    "trip_id",
    F.sha2(
        F.concat_ws(
            "|",
            F.col("vendor_id").cast("string"),
            F.col("pickup_datetime").cast("string"),
            F.col("dropoff_datetime").cast("string"),
            F.col("fare_amount").cast("string"),
            F.col("trip_distance_miles").cast("string"),
        ),
        256,
    )
)

    # Foreign key -> dim_date
    .withColumn(
        "date_key",
        F.date_format(
            F.to_date("pickup_datetime"),
            "yyyyMMdd"
        ).cast("int")
    )

    # Feature engineering
    .withColumn(
        "duration_minutes",
        (
            F.unix_timestamp("dropoff_datetime")
            - F.unix_timestamp("pickup_datetime")
        ) / 60
    )

    .withColumn(
        "fare_per_mile",
        F.when(
            F.col("trip_distance_miles") > 0,
            F.col("fare_amount") / F.col("trip_distance_miles")
        )
    )

    .withColumn(
        "tip_percentage",
        F.when(
            F.col("fare_amount") > 0,
            F.col("tip_amount") / F.col("fare_amount") * 100
        )
    )

    .withColumn(
        "pickup_hour",
        F.hour("pickup_datetime")
    )

    .withColumn(
        "is_rush_hour",
        (
            F.col("pickup_hour").between(7, 9)
            | F.col("pickup_hour").between(16, 19)
        )
    )

    .select(
        "trip_id",
        "date_key",
        "vendor_id",
        "payment_type",
        "rate_code_id",
        "passenger_count",
        "trip_distance_miles",
        "fare_amount",
        "extra_amount",
        "mta_tax_amount",
        "tip_amount",
        "tolls_amount",
        "improvement_surcharge",
        "total_amount",
        "pickup_datetime",
        "dropoff_datetime",
        "pickup_hour",
        "duration_minutes",
        "fare_per_mile",
        "tip_percentage",
        "is_rush_hour",
        "pickup_longitude",
        "pickup_latitude",
        "dropoff_longitude",
        "dropoff_latitude"
    )
)

---
# Save data to Gold Layer

In [10]:
dim_date.write.mode("overwrite").parquet(
    "../data/gold/dim_date/"
)

dim_vendor.write.mode("overwrite").parquet(
    "../data/gold/dim_vendor/"
)

dim_payment.write.mode("overwrite").parquet(
    "../data/gold/dim_payment/"
)

dim_rate_code.write.mode("overwrite").parquet(
    "../data/gold/dim_rate_code/"
)

fact_trips.write.mode("overwrite").parquet(
    "../data/gold/fact_trips/"
)

# Show data

In [11]:
fact_trips = spark.read.parquet("../data/gold/fact_trips/")
dim_date = spark.read.parquet("../data/gold/dim_date/")
dim_vendor = spark.read.parquet("../data/gold/dim_vendor/")
dim_payment = spark.read.parquet("../data/gold/dim_payment/")
dim_rate_code = spark.read.parquet("../data/gold/dim_rate_code/")

---

## Schema fact_trips

In [12]:
print("-fact_trips")
fact_trips.printSchema()
print("-dim_date")
dim_date.printSchema()
print("-dim_vendor")
dim_vendor.printSchema()
print("-dim_payment")
dim_payment.printSchema()
print("-dim_rate_code")
dim_rate_code.printSchema()

-fact_trips
root
 |-- trip_id: string (nullable = true)
 |-- date_key: integer (nullable = true)
 |-- vendor_id: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- rate_code_id: integer (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance_miles: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra_amount: double (nullable = true)
 |-- mta_tax_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- duration_minutes: double (nullable = true)
 |-- fare_per_mile: double (nullable = true)
 |-- tip_percentage: double (nullable = true)
 |-- is_rush_hour: boolean (nullable = true)
 |-- pickup_longitude: doub

In [20]:
fact_trips.show(5, truncate=False)

+----------------------------------------------------------------+--------+---------+------------+------------+---------------+-------------------+-----------+------------+--------------+----------+------------+---------------------+------------+-------------------+-------------------+-----------+------------------+------------------+------------------+------------+------------------+-----------------+------------------+------------------+
|trip_id                                                         |date_key|vendor_id|payment_type|rate_code_id|passenger_count|trip_distance_miles|fare_amount|extra_amount|mta_tax_amount|tip_amount|tolls_amount|improvement_surcharge|total_amount|pickup_datetime    |dropoff_datetime   |pickup_hour|duration_minutes  |fare_per_mile     |tip_percentage    |is_rush_hour|pickup_longitude  |pickup_latitude  |dropoff_longitude |dropoff_latitude  |
+----------------------------------------------------------------+--------+---------+------------+------------+-

In [18]:
dim_date.show(10, truncate=False)

+----------+--------+----+-----+---+-------+-------+----------+
|full_date |date_key|year|month|day|weekday|quarter|is_weekend|
+----------+--------+----+-----+---+-------+-------+----------+
|2016-02-04|20160204|2016|2    |4  |5      |1      |false     |
|2016-02-08|20160208|2016|2    |8  |2      |1      |false     |
|2016-02-03|20160203|2016|2    |3  |4      |1      |false     |
|2016-02-22|20160222|2016|2    |22 |2      |1      |false     |
|2016-02-11|20160211|2016|2    |11 |5      |1      |false     |
|2016-02-09|20160209|2016|2    |9  |3      |1      |false     |
|2016-02-15|20160215|2016|2    |15 |2      |1      |false     |
|2016-02-25|20160225|2016|2    |25 |5      |1      |false     |
|2016-02-18|20160218|2016|2    |18 |5      |1      |false     |
|2016-02-28|20160228|2016|2    |28 |1      |1      |true      |
+----------+--------+----+-----+---+-------+-------+----------+
only showing top 10 rows


In [16]:
dim_vendor.show(truncate=False)

+---------+----------------------------+
|vendor_id|vendor_name                 |
+---------+----------------------------+
|1        |Creative Mobile Technologies|
|2        |VeriFone Inc                |
+---------+----------------------------+



In [17]:
dim_payment.show(truncate=False)

+------------+------------+
|payment_type|payment_name|
+------------+------------+
|1           |Credit Card |
|3           |No Charge   |
|4           |Dispute     |
|2           |Cash        |
+------------+------------+



In [15]:
dim_rate_code.show(truncate=False)

+------------+---------------------+
|rate_code_id|rate_code_name       |
+------------+---------------------+
|1           |Standard Rate        |
|6           |Group Ride           |
|3           |Newark               |
|5           |Negotiated Fare      |
|4           |Nassau or Westchester|
|2           |JFK                  |
|99          |Other                |
+------------+---------------------+



In [14]:
# spark.stop()